# Batch Evaluation using Gemini Batch API
This notebook performs large-scale flaky test classification using the Gemini Batch API.

Instead of sending one API request per test case, this notebook creates JSONL batch request files and executes multiple experiments asynchronously.

## Workflow

The evaluation pipeline consists of the following stages:

1. Configure the project environment
2. Load the evaluation dataset
3. Generate prompts
4. Create Gemini Batch requests
5. Upload JSONL files
6. Execute Batch jobs
7. Download predictions
8. Validate responses
9. Save structured results

In [2]:
from pathlib import Path
import sys

# Project root directory
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python's module search path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:")
print(PROJECT_ROOT)

Project Root:
d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework


In [3]:
import json
import time

import pandas as pd
from tqdm import tqdm

from google import genai
from google.genai import types

In [4]:
from utils.prompt_builder import build_prompt

print("✓ Prompt builder imported successfully")

✓ Prompt builder imported successfully


## Project Configuration

This section defines the project directories used throughout the evaluation pipeline.

The notebook stores:

- Input datasets
- Batch request JSONL files
- Raw Batch API responses
- Processed prediction results

Keeping these paths centralized improves maintainability and reproducibility.

In [5]:
# ==========================================
# Project Directories
# ==========================================

DATA_DIR = PROJECT_ROOT / "datasets"

BATCH_REQUEST_DIR = PROJECT_ROOT / "generated_prompts" / "batch_requests"

RAW_RESULTS_DIR = PROJECT_ROOT / "results" / "raw_batch_results"

RESULTS_DIR = PROJECT_ROOT / "results"

In [6]:
# Create output directories

BATCH_REQUEST_DIR.mkdir(parents=True, exist_ok=True)
RAW_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directories are ready.")

Project directories are ready.


In [7]:
print("Data Directory         :", DATA_DIR)
print("Batch Requests         :", BATCH_REQUEST_DIR)
print("Raw Results            :", RAW_RESULTS_DIR)
print("Processed Results      :", RESULTS_DIR)

Data Directory         : d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\datasets
Batch Requests         : d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\generated_prompts\batch_requests
Raw Results            : d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\raw_batch_results
Processed Results      : d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results


## Configure Gemini API

This section initializes the Gemini client used throughout the evaluation pipeline.

The client authenticates using a Google AI Studio API key and is reused for all Batch API operations, including file uploads, batch job creation, status monitoring, and result downloads.

In [ ]:
# ==========================================
# Gemini API Configuration
# ==========================================

GEMINI_API_KEY = ""

client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("✓ Gemini client initialized successfully.")

✓ Gemini client initialized successfully.


## Load Evaluation Dataset

This section loads the evaluation dataset that will be used for Batch API inference.

Before generating prompts, the dataset is inspected to verify:

- The dataset was loaded successfully.
- The number of test cases is correct.
- The required columns for prompt generation are available.

In [9]:
# ==========================================
# Load Evaluation Dataset
# ==========================================

DATASET_PATH = DATA_DIR / "evaluation_dataset.jsonl"

df = pd.read_json(DATASET_PATH, lines=True)

print(f"Dataset loaded successfully.")
print(f"Total samples: {len(df)}")

Dataset loaded successfully.
Total samples: 2210


In [10]:
df.head()

,id,test_id,isFlaky,issue_category,repo_url,issue_commit,fixed_commit,test_code,helper_methods_json,failure_log,code_under_test_json,test_code_original,helper_methods_json_original,failure_log_original,code_under_test_original,has_helper_methods,has_code_under_test,has_failure_log,context_score
0,857,ormlitecore59309e55,True,Order Dependent,https://github.com/j256/ormlite-core,59309e51c61e8a63cb5fd24a5a7607b668a5f095,c80bde196ca152ecc8a3c4f38f77dbe5a4ea3232,@Test\n\tpublic void testSetObjectCacheThrow()...,{},org.opentest4j.AssertionFailedError: Unexpecte...,{'com.j256.ormlite.dao.BaseDaoImpl': {'setObje...,@Test\n\tpublic void testSetObjectCacheThrow()...,{},Failed Rounds: 10/10\norg.opentest4j.Assertion...,{'com.j256.ormlite.dao.BaseDaoImpl': {'setObje...,False,True,True,3
1,1574,Closure-144-21,False,Non-Flaky,https://github.com/google/closure-compiler,c9e89727dc8063d087d28e42629606f4fd74a6e5,465282f1ca28a208b06c47b55fd292d4631c55da,public void testExportMultiple3() throws Excep...,{'compileAndCheck': 'private void compileAndCh...,junit.framework.ComparisonFailure: expected:<....,{'com.google.javascript.jscomp.Result': {'<ini...,public void testExportMultiple3() throws Excep...,{'compileAndCheck': 'private void compileAndCh...,Failed Rounds: 1/1\njunit.framework.Comparison...,{'com.google.javascript.jscomp.Result': {'<ini...,True,True,True,5
2,1483,Closure-115-5,False,Non-Flaky,https://github.com/google/closure-compiler,2d6e1c78f41248fbbb1eec43b23e7430e2bc7885,4597738e8898f738c1f969fe90479728be81cc80,public void testInlineFunctions6() {\n\n te...,"{'test': 'public void test(String js, String e...",junit.framework.AssertionFailedError:\nExpecte...,{'com.google.javascript.jscomp.DiagnosticType'...,public void testInlineFunctions6() {\n // m...,{'test': '/** * Verifies that the compiler ...,Failed Rounds: 1/1\njunit.framework.AssertionF...,{'com.google.javascript.jscomp.DiagnosticType'...,True,True,True,5
3,431,ignite3modulesstoragerocksdb19c8a82testAbortWrite,True,Implementation Dependent,https://github.com/apache/ignite-3,19c8a824bd9d31f0d0dbd3fbbdd2a32ee360cab2,de6ee0702398f9ce3022a8e265c633857f3a3d88,.\n */\n @Test\n public void testAbo...,{'read': 'protected BinaryRow read(RowId rowId...,org.junit.jupiter.api.extension.ParameterResol...,{'org.apache.ignite.internal.hlc.HybridTimesta...,.\n */\n @Test\n public void testAbo...,{'read': '/** * Reads a row. */ ...,Failed Rounds: 84/101\norg.junit.jupiter.api.e...,{'org.apache.ignite.internal.hlc.HybridTimesta...,True,True,True,5
4,1563,Closure-144-10,False,Non-Flaky,https://github.com/google/closure-compiler,c9e89727dc8063d087d28e42629606f4fd74a6e5,465282f1ca28a208b06c47b55fd292d4631c55da,public void testVariableArgumentsTypesAnnotati...,{'assertTypeAnnotations': 'private void assert...,junit.framework.ComparisonFailure: expected:<....,{'com.google.javascript.rhino.Node': {'getDoub...,public void testVariableArgumentsTypesAnnotati...,{'assertTypeAnnotations': 'private void assert...,Failed Rounds: 1/1\njunit.framework.Comparison...,{'com.google.javascript.rhino.Node': {'getDoub...,True,True,True,5


In [11]:
print("Dataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

Dataset Shape:
(2210, 19)

Columns:
['id', 'test_id', 'isFlaky', 'issue_category', 'repo_url', 'issue_commit', 'fixed_commit', 'test_code', 'helper_methods_json', 'failure_log', 'code_under_test_json', 'test_code_original', 'helper_methods_json_original', 'failure_log_original', 'code_under_test_original', 'has_helper_methods', 'has_code_under_test', 'has_failure_log', 'context_score']


In [12]:
print(df.columns.tolist())

['id', 'test_id', 'isFlaky', 'issue_category', 'repo_url', 'issue_commit', 'fixed_commit', 'test_code', 'helper_methods_json', 'failure_log', 'code_under_test_json', 'test_code_original', 'helper_methods_json_original', 'failure_log_original', 'code_under_test_original', 'has_helper_methods', 'has_code_under_test', 'has_failure_log', 'context_score']


## Validate Dataset Schema

Before generating prompts, the dataset schema is validated to ensure all required fields are available.

This validation prevents runtime errors caused by missing columns and ensures compatibility with the prompt generation utility.

In [13]:
# ==========================================
# Validate Required Columns
# ==========================================

REQUIRED_COLUMNS = [
    "id",
    "test_code",
    "helper_methods_json",
    "code_under_test_json",
    "failure_log",
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Dataset is missing required columns: {missing_columns}"
    )

print("✓ Dataset schema validated successfully.")

✓ Dataset schema validated successfully.


## Generate Batch Request Files

This section converts each evaluation sample into a Gemini Batch API request.

For every experiment configuration, the notebook:

1. Generates a prompt using the existing `build_prompt()` utility.
2. Wraps the prompt in the Gemini Batch API request format.
3. Writes one JSON object per line into a JSONL file.

Each experiment produces a separate batch request file, which can later be uploaded to the Gemini Batch API.

In [44]:
# ==========================================
# Gemini Model Configuration
# ==========================================

MODEL_NAME = "gemini-3.1-flash-lite"

print(f"Model: {MODEL_NAME}")

Model: gemini-3.1-flash-lite


### Batch Request Format

Gemini Batch API expects every request to be represented as a single JSON object.

Each object contains:

- A unique request identifier.
- The target model.
- The generated prompt.

These objects are written line-by-line into a JSONL file.

### Create a Batch Request

Each evaluation sample is converted into a single Gemini Batch API request.

The request consists of:

- **key** – A unique identifier for matching responses back to the original dataset.
- **request** – A valid `GenerateContentRequest` containing the generated prompt and model generation configuration.

Each request will later become one line in the JSONL input file submitted to the Batch API.

In [15]:
import json


def create_batch_request(sample, strategy, include_context):

    #Convert a dataset sample into a Gemini Batch API request.


    prompt = build_prompt(
        sample=sample,
        strategy=strategy,
        include_context=include_context,
    )

    return {
        "key": str(sample["id"]),
        "request": {
            "contents": [
                {
                    "parts": [
                        {
                            "text": prompt
                        }
                    ]
                }
            ],
            "generationConfig": {
                "temperature": 0.0,
                "responseMimeType": "application/json"
            }
        }
    }

### Verify Batch Request Generation

Before generating the full batch request files, a single dataset sample is converted into a batch request and inspected.

This verification ensures that the generated JSON structure conforms to the Gemini Batch API specification.

In [16]:
sample = df.iloc[0]

request = create_batch_request(
    sample=sample,
    strategy="zero_shot",
    include_context=False
)

print(json.dumps(request, indent=2))

{
  "key": "857",
  "request": {
    "contents": [
      {
        "parts": [
          {
            "text": "You are an expert software testing assistant specializing in flaky test identification and classification.\n\nBase every decision solely on the provided artifacts.\n\nDo not rely on external knowledge or assumptions.\n\nIf the available evidence is insufficient to confidently identify a flaky test, classify it as Non-Flaky.\n\nYour task is to analyze the provided software testing artifacts.\n\nDetermine whether the test is Flaky or Non-Flaky.\n\nIf the test is Flaky, select exactly one issue category from the provided category list.\n\nJustify the classification using only the provided evidence.\n\nIdentify the artifacts that support your conclusion.\n\n## Flaky Test Categories\n\nIf the test is classified as Flaky, select exactly one issue category from the following list.\n\n1. Implementation Dependent\n   - The test outcome depends on implementation-specific behavior rather

## Generate Batch Request Files

This section generates one Gemini Batch API input file for each experiment.

For every experiment:

1. Iterate through all dataset samples.
2. Generate a prompt using `build_prompt()`.
3. Convert it into a Gemini Batch API request.
4. Save each request as one JSON object per line in a JSONL file.

Each experiment produces a separate JSONL file that can be submitted independently to the Gemini Batch API.

In [17]:
EXPERIMENTS = [
    {
        "strategy": "zero_shot",
        "include_context": False,
    },
    {
        "strategy": "zero_shot",
        "include_context": True,
    },
    {
        "strategy": "zero_shot_cot",
        "include_context": False,
    },
    {
        "strategy": "zero_shot_cot",
        "include_context": True,
    },
    {
        "strategy": "few_shot_cot",
        "include_context": False,
    },
    {
        "strategy": "few_shot_cot",
        "include_context": True,
    },
]

In [18]:
import json

for experiment in EXPERIMENTS:

    strategy = experiment["strategy"]
    include_context = experiment["include_context"]

    experiment_name = (
        f"{strategy}_{'with_context' if include_context else 'without_context'}"
    )

    output_path = BATCH_REQUEST_DIR / f"{experiment_name}.jsonl"

    print(f"Generating {output_path.name}...")

    with open(output_path, "w", encoding="utf-8") as f:

        for _, sample in df.iterrows():

            request = create_batch_request(
                sample=sample,
                strategy=strategy,
                include_context=include_context
            )

            f.write(json.dumps(request))
            f.write("\n")

    print(f"✓ Saved {output_path.name}")

Generating zero_shot_without_context.jsonl...
✓ Saved zero_shot_without_context.jsonl
Generating zero_shot_with_context.jsonl...
✓ Saved zero_shot_with_context.jsonl
Generating zero_shot_cot_without_context.jsonl...
✓ Saved zero_shot_cot_without_context.jsonl
Generating zero_shot_cot_with_context.jsonl...
✓ Saved zero_shot_cot_with_context.jsonl
Generating few_shot_cot_without_context.jsonl...
✓ Saved few_shot_cot_without_context.jsonl
Generating few_shot_cot_with_context.jsonl...
✓ Saved few_shot_cot_with_context.jsonl


### Validate Generated Batch Files

Verify that every line in every generated JSONL file is valid JSON and contains the required Gemini Batch API fields.

In [19]:
import json

required_keys = {
    "key",
    "request",
}

required_request_keys = {
    "contents",
}

for file in sorted(BATCH_REQUEST_DIR.glob("*.jsonl")):

    with open(file, encoding="utf-8") as f:

        for line_no, line in enumerate(f, start=1):

            obj = json.loads(line)

            assert required_keys <= obj.keys(), \
                f"{file.name} line {line_no}: Missing top-level keys"

            assert required_request_keys <= obj["request"].keys(), \
                f"{file.name} line {line_no}: Missing request fields"

    print(f"✓ {file.name}")

✓ few_shot_cot_with_context.jsonl
✓ few_shot_cot_without_context.jsonl
✓ zero_shot_cot_with_context.jsonl
✓ zero_shot_cot_without_context.jsonl
✓ zero_shot_with_context.jsonl
✓ zero_shot_without_context.jsonl


In [20]:
for file in sorted(BATCH_REQUEST_DIR.glob("*.jsonl")):

    with open(file, encoding="utf-8") as f:
        count = sum(1 for _ in f)

    print(f"{file.name}: {count}")

few_shot_cot_with_context.jsonl: 2210
few_shot_cot_without_context.jsonl: 2210
zero_shot_cot_with_context.jsonl: 2210
zero_shot_cot_without_context.jsonl: 2210
zero_shot_with_context.jsonl: 2210
zero_shot_without_context.jsonl: 2210


In [21]:
import json

with open(BATCH_REQUEST_DIR / "few_shot_cot_with_context.jsonl", encoding="utf-8") as f:

    ids = [json.loads(line)["key"] for line in f]

print("Total IDs :", len(ids))
print("Unique IDs:", len(set(ids)))

Total IDs : 2210
Unique IDs: 2210


In [22]:
dataset_ids = set(df["id"].astype(str))

generated_ids = set(ids)

print(dataset_ids == generated_ids)

True


In [23]:
import random
import json

indices = random.sample(range(len(df)), 5)

with open(BATCH_REQUEST_DIR / "few_shot_cot_with_context.jsonl", encoding="utf-8") as f:
    requests = [json.loads(line) for line in f]

for i in indices:

    sample = df.iloc[i]

    prompt = requests[i]["request"]["contents"][0]["parts"][0]["text"]

    print("="*80)
    print("Dataset ID:", sample["id"])
    print("Request Key:", requests[i]["key"])

    print("\nDataset Test Code (first 150 chars):")
    print(sample["test_code"][:150])

    print("\nFound in Prompt?")
    print(sample["test_code"][:100] in prompt)

Dataset ID: 1181
Request Key: 1181

Dataset Test Code (first 150 chars):
public void testRemoveGlobal3() {
    removeGlobal = false;
    testSame("var x=1");
    test("function x(){function y(x){var z;}y()}",
         "func

Found in Prompt?
True
Dataset ID: 430
Request Key: 430

Dataset Test Code (first 150 chars):
@Test
    void secondUncommittedWriteWithSameTxIdReplacesExistingUncommittedWrite() {
        RowId rowId = insert(binaryRow, txId);

        addWrite

Found in Prompt?
True
Dataset ID: 1418
Request Key: 1418

Dataset Test Code (first 150 chars):
public void testIssue297() {
    args.add("--compilation_level=SIMPLE_OPTIMIZATIONS");
    test("function f(p) {" +
         " var x;" +
         " re

Found in Prompt?
True
Dataset ID: 1619
Request Key: 1619

Dataset Test Code (first 150 chars):
public void testFunctionInference16() throws Exception {
    testFunctionType(
        "/** @constructor */ function f() {};" +
        "f.prototype.f

Found in Prompt?
True
Dataset ID: 17

In [24]:
import hashlib
import json

hashes = []

with open(BATCH_REQUEST_DIR / "few_shot_cot_with_context.jsonl", encoding="utf-8") as f:

    for line in f:

        prompt = json.loads(line)["request"]["contents"][0]["parts"][0]["text"]

        hashes.append(hashlib.md5(prompt.encode()).hexdigest())

print("Total prompts :", len(hashes))
print("Unique prompts:", len(set(hashes)))

Total prompts : 2210
Unique prompts: 2169


In [25]:
import json

with open(BATCH_REQUEST_DIR / "few_shot_cot_with_context.jsonl", encoding="utf-8") as f:
    requests = [json.loads(line) for line in f]

for i in [0, 500, 1000, 1500, 2209]:

    sample = df.iloc[i]

    prompt = requests[i]["request"]["contents"][0]["parts"][0]["text"]

    assert sample["test_code"] in prompt

print("✓ All selected samples matched.")

✓ All selected samples matched.


## Create a Pilot Dataset

Before evaluating the complete dataset, we create a small pilot dataset.

The pilot dataset allows us to verify that the entire Gemini Batch API workflow works correctly while consuming minimal API quota.

Once the pilot succeeds, the same workflow can be executed on the complete dataset without any code changes.

In [26]:
# Number of samples for pilot evaluation
PILOT_SIZE = 10

# Fixed random seed for reproducibility
pilot_df = df.sample(
    n=PILOT_SIZE,
    random_state=42
).reset_index(drop=True)

print(f"Pilot dataset contains {len(pilot_df)} samples.")

pilot_df[["id"]]

Pilot dataset contains 10 samples.


,id
0,1717
1,1938
2,358
3,199
4,2400
5,2233
6,1811
7,1790
8,628
9,294


## Generate Pilot Batch Request File

Generate a Gemini Batch API request file using only the pilot dataset.

This file will be used to verify the complete evaluation pipeline before processing the full dataset.

In [27]:
import json

pilot_file = BATCH_REQUEST_DIR / "pilot_few_shot_cot_with_context.jsonl"

with open(pilot_file, "w", encoding="utf-8") as f:

    for _, sample in pilot_df.iterrows():

        request = create_batch_request(
            sample=sample,
            strategy="few_shot_cot",
            include_context=True,
        )

        f.write(json.dumps(request))
        f.write("\n")

print("Pilot batch file created:")
print(pilot_file)

Pilot batch file created:
d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\generated_prompts\batch_requests\pilot_few_shot_cot_with_context.jsonl


In [28]:
import json

with open(pilot_file, encoding="utf-8") as f:
    requests = [json.loads(line) for line in f]

print("Number of requests:", len(requests))

for request in requests[:3]:
    print(request["key"])

Number of requests: 10
1717
1938
358


## Upload Pilot Batch File

Upload the pilot JSONL file to the Gemini File API.

The uploaded file will be used as the input source for the Gemini Batch API job.

In [31]:
from google.genai import types

uploaded_pilot_file = client.files.upload(
    file=str(pilot_file),
    config=types.UploadFileConfig(
        display_name="pilot-few-shot-cot-with-context",
        mime_type="application/jsonl",
    ),
)

print("Upload completed!")
print(uploaded_pilot_file.name)

Upload completed!
files/h9dnpuypus9t


## Create Gemini Batch Job

Create a Gemini Batch API job using the uploaded pilot request file.

The returned batch job can be monitored until processing is complete.

In [45]:
pilot_batch_job = client.batches.create(
    model=MODEL_NAME,
    src=uploaded_pilot_file.name,
    config={
        "display_name": "pilot-few-shot-cot-with-context"
    },
)

print("Batch job created!")
print("Job Name :", pilot_batch_job.name)
print("State    :", pilot_batch_job.state)

Batch job created!
Job Name : batches/uk0bm8p90zaj6oyjpmbjy3wl3lkwab9dvhs1
State    : JobState.JOB_STATE_PENDING


## Step 10 - Wait for Batch Completion

A batch job executes asynchronously.

This cell periodically checks the batch status until it reaches one of the following terminal states:

- JOB_STATE_SUCCEEDED
- JOB_STATE_FAILED
- JOB_STATE_CANCELLED

After the batch succeeds, the output file can be downloaded for processing.

In [46]:
import time

while True:
    pilot_batch_job = client.batches.get(name=pilot_batch_job.name)

    print(f"Current State: {pilot_batch_job.state}")

    if pilot_batch_job.state.name in [
        "JOB_STATE_SUCCEEDED",
        "JOB_STATE_FAILED",
        "JOB_STATE_CANCELLED",
    ]:
        break

    time.sleep(10)

print("\nFinal State:", pilot_batch_job.state)

Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_RUNNING
Current State: JobState.JOB_STATE_SUCCEEDED

Final State: JobState.JOB_STATE_SUCCEEDED


## Step 11 - Inspect Completed Batch

After the batch finishes successfully, inspect the metadata to obtain the generated output file.

This output file contains the model predictions in JSONL format and will be downloaded in the next step.

In [47]:
print("Batch Name:")
print(pilot_batch_job.name)

print("\nState:")
print(pilot_batch_job.state)

print("\nDestination:")
print(pilot_batch_job.dest)

Batch Name:
batches/uk0bm8p90zaj6oyjpmbjy3wl3lkwab9dvhs1

State:
JobState.JOB_STATE_SUCCEEDED

Destination:
format=None gcs_uri=None bigquery_uri=None file_name='files/batch-uk0bm8p90zaj6oyjpmbjy3wl3lkwab9dvhs1' inlined_responses=None inlined_embed_content_responses=None vertex_dataset=None


## Step 12 - Download Batch Results

Download the output JSONL file produced by the completed batch job.

The downloaded file will be stored locally for parsing and evaluation.

In [48]:
output_file = client.files.download(
    file=pilot_batch_job.dest.file_name
)

print(output_file)

b'{"response":{"candidates":[{"content":{"parts":[{"text":"{\\n    \\"classification\\": \\"Non-Flaky\\",\\n    \\"category\\": \\"Non-Flaky\\",\\n    \\"reasoning\\": \\"The test performs a static transformation of a JavaScript string and compares the resulting AST against an expected output. The failure is a clear mismatch between the expected tree and the actual tree produced by the compiler. There is no evidence of concurrency, timing dependencies, or shared state that would cause the test to pass in some runs and fail in others. The failure is deterministic, indicating a bug in the compiler\'s optimization logic (specifically in function inlining) rather than flakiness.\\",\\n    \\"evidence\\": [\\n        \\"Test Code\\",\\n        \\"Helper Methods\\",\\n        \\"Failure Log\\"\\n    ]\\n}","thoughtSignature":"EjQKMgERTTIPE80iZZnQD8QM3bRmR2yTjig9mW84a37g1aisZJ8fhCAiwWHmcfpyHhsdQUHT"}],"role":"model"},"finishReason":"STOP","index":0}],"usageMetadata":{"promptTokenCount":5821,"

## Step 13 - Create Experiment Result Directory

Create the directory structure where both the raw Gemini response and the processed prediction file will be stored.

Folder structure:

results/
└── <model>/
    └── <prompt_type>/
        └── <with_context|without_context>/

In [50]:
from pathlib import Path

# Experiment metadata
MODEL_FOLDER = "gemini-3.1-flash-lite"
PROMPT_TYPE = "few_shot_cot"
CONTEXT_ENABLED = True

context_folder = "with_context" if CONTEXT_ENABLED else "without_context"

experiment_dir = (
    RESULTS_DIR
    / MODEL_FOLDER
    / PROMPT_TYPE
    / context_folder
)

experiment_dir.mkdir(parents=True, exist_ok=True)

print("Experiment folder:")
print(experiment_dir)

Experiment folder:
d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gemini-3.1-flash-lite\few_shot_cot\with_context


## Step 14 - Save Pilot Batch Response

Download the completed pilot batch output from Gemini and save the raw JSONL response.

This raw response will be used in the next step to extract the model predictions and generate the processed predictions file.

In [51]:
# Download the completed pilot batch output
output_file = client.files.download(
    file=pilot_batch_job.dest.file_name
)

# Save the raw response
pilot_raw_response_path = experiment_dir / "pilot_raw_response.jsonl"

with open(pilot_raw_response_path, "wb") as f:
    f.write(output_file)

print(f"Pilot raw response saved to:\n{pilot_raw_response_path}")

Pilot raw response saved to:
d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gemini-3.1-flash-lite\few_shot_cot\with_context\pilot_raw_response.jsonl


## Step 15 - Read the Pilot Raw Response

Load the saved pilot batch response from disk.

Since the Gemini Batch API returns newline-delimited JSON (JSONL), each line represents the response for one input sample.

In this step, we only read the file and convert each line into a Python dictionary.

In [52]:
import json

pilot_records = []

with open(pilot_raw_response_path, "r", encoding="utf-8") as f:
    for line in f:
        pilot_records.append(json.loads(line))

print(f"Loaded {len(pilot_records)} records.")

Loaded 10 records.


In [53]:
print(type(pilot_records))
print(type(pilot_records[0]))
print()

print(pilot_records[0].keys())

<class 'list'>
<class 'dict'>

dict_keys(['response', 'key'])


In [54]:
import pprint

pprint.pprint(pilot_records[0]["response"], width=120)

{'candidates': [{'content': {'parts': [{'text': '{\n'
                                                '    "classification": "Non-Flaky",\n'
                                                '    "category": "Non-Flaky",\n'
                                                '    "reasoning": "The test performs a static transformation of a '
                                                'JavaScript string and compares the resulting AST against an expected '
                                                'output. The failure is a clear mismatch between the expected tree and '
                                                'the actual tree produced by the compiler. There is no evidence of '
                                                'concurrency, timing dependencies, or shared state that would cause '
                                                'the test to pass in some runs and fail in others. The failure is '
                                                "deterministic, indica

In [55]:
print(pilot_records[0]["response"].keys())

dict_keys(['candidates', 'usageMetadata', 'modelVersion', 'responseId'])


In [56]:
print(pilot_records[0]["response"]["candidates"][0].keys())

dict_keys(['content', 'finishReason', 'index'])


In [57]:
print(pilot_records[0]["response"]["candidates"][0]["content"].keys())

dict_keys(['parts', 'role'])


In [58]:
print(pilot_records[0]["response"]["candidates"][0]["content"]["parts"])

[{'text': '{\n    "classification": "Non-Flaky",\n    "category": "Non-Flaky",\n    "reasoning": "The test performs a static transformation of a JavaScript string and compares the resulting AST against an expected output. The failure is a clear mismatch between the expected tree and the actual tree produced by the compiler. There is no evidence of concurrency, timing dependencies, or shared state that would cause the test to pass in some runs and fail in others. The failure is deterministic, indicating a bug in the compiler\'s optimization logic (specifically in function inlining) rather than flakiness.",\n    "evidence": [\n        "Test Code",\n        "Helper Methods",\n        "Failure Log"\n    ]\n}', 'thoughtSignature': 'EjQKMgERTTIPE80iZZnQD8QM3bRmR2yTjig9mW84a37g1aisZJ8fhCAiwWHmcfpyHhsdQUHT'}]


## Step 17 - Parse Gemini Predictions

Each Gemini response contains a JSON string inside
`response → candidates → content → parts → text`.

This step extracts the JSON string and converts it into a Python dictionary that can be used to generate the final predictions file.

In [59]:
import json

parsed_predictions = []

for record in pilot_records:
    # Extract the sample ID
    sample_id = int(record["key"])

    # Extract Gemini's JSON response
    response_text = (
        record["response"]
        ["candidates"][0]
        ["content"]
        ["parts"][0]
        ["text"]
    )

    # Convert JSON string to Python dictionary
    prediction = json.loads(response_text)

    parsed_predictions.append({
        "id": sample_id,
        "prediction": prediction
    })

print(f"Parsed {len(parsed_predictions)} predictions.")

Parsed 10 predictions.


In [60]:
parsed_predictions[0]

{'id': 1717,
 'prediction': {'classification': 'Non-Flaky',
  'category': 'Non-Flaky',
  'reasoning': "The test performs a static transformation of a JavaScript string and compares the resulting AST against an expected output. The failure is a clear mismatch between the expected tree and the actual tree produced by the compiler. There is no evidence of concurrency, timing dependencies, or shared state that would cause the test to pass in some runs and fail in others. The failure is deterministic, indicating a bug in the compiler's optimization logic (specifically in function inlining) rather than flakiness.",
  'evidence': ['Test Code', 'Helper Methods', 'Failure Log']}}

## Step 18 - Create Processed Predictions

Merge the parsed Gemini predictions with the original dataset to create a processed predictions file.

Each record contains:

- Original test metadata
- Ground truth labels
- Model predictions
- Experiment metadata

The processed predictions file follows the same schema used in previous experiments, enabling a common evaluation pipeline across all models.

In [62]:
print(df.columns.tolist())

['id', 'test_id', 'isFlaky', 'issue_category', 'repo_url', 'issue_commit', 'fixed_commit', 'test_code', 'helper_methods_json', 'failure_log', 'code_under_test_json', 'test_code_original', 'helper_methods_json_original', 'failure_log_original', 'code_under_test_original', 'has_helper_methods', 'has_code_under_test', 'has_failure_log', 'context_score']


In [65]:
# Create a lookup table from the original dataset
df_lookup = df.set_index("id")

processed_predictions = []

for item in parsed_predictions:
    sample_id = item["id"]
    prediction = item["prediction"]

    sample = df_lookup.loc[sample_id]

    processed_predictions.append({
        "id": sample_id,
        "test_id": sample["test_id"],

        # Convert boolean -> string
        "ground_truth_classification": "Flaky" if sample["isFlaky"] else "Non-Flaky",
        "ground_truth_category": sample["issue_category"],

        # Gemini predictions
        "predicted_classification": prediction["classification"],
        "predicted_category": prediction["category"],

        # Explanation
        "reasoning": prediction["reasoning"],
        "evidence": prediction["evidence"],

        # Experiment metadata
        "model": MODEL_FOLDER,
        "prompt_type": PROMPT_TYPE,
        "context_enabled": CONTEXT_ENABLED,

        # Gemini Batch API doesn't provide per-request latency
        "latency_ms": None
    })

print(f"Created {len(processed_predictions)} prediction records.")

Created 10 prediction records.


In [66]:
df[["isFlaky", "issue_category"]].head()

,isFlaky,issue_category
0,True,Order Dependent
1,False,Non-Flaky
2,False,Non-Flaky
3,True,Implementation Dependent
4,False,Non-Flaky


## Step 19 - Save Processed Predictions

Save the processed predictions as a JSONL file.

This file contains the ground truth labels, Gemini predictions, reasoning, and experiment metadata. It serves as the input for the evaluation stage.

In [67]:
pilot_predictions_path = experiment_dir / "pilot_predictions.jsonl"

with open(pilot_predictions_path, "w", encoding="utf-8") as f:
    for record in processed_predictions:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(processed_predictions)} predictions")
print(pilot_predictions_path)

Saved 10 predictions
d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gemini-3.1-flash-lite\few_shot_cot\with_context\pilot_predictions.jsonl


In [68]:
with open(pilot_predictions_path, "r", encoding="utf-8") as f:
    print(f.readline())

{"id": 1717, "test_id": "Closure-175-5", "ground_truth_classification": "Non-Flaky", "ground_truth_category": "Non-Flaky", "predicted_classification": "Non-Flaky", "predicted_category": "Non-Flaky", "reasoning": "The test performs a static transformation of a JavaScript string and compares the resulting AST against an expected output. The failure is a clear mismatch between the expected tree and the actual tree produced by the compiler. There is no evidence of concurrency, timing dependencies, or shared state that would cause the test to pass in some runs and fail in others. The failure is deterministic, indicating a bug in the compiler's optimization logic (specifically in function inlining) rather than flakiness.", "evidence": ["Test Code", "Helper Methods", "Failure Log"], "model": "gemini-3.1-flash-lite", "prompt_type": "few_shot_cot", "context_enabled": true, "latency_ms": null}



## Upload Batch Request Files

Upload each generated JSONL request file to the Gemini Files API.

Each uploaded file will later be used to create a separate Batch API job for one experimental configuration.

The uploaded file objects are stored in a dictionary keyed by the experiment name, allowing the remaining pipeline to reference them easily.

In [69]:
uploaded_files = {}

for experiment in EXPERIMENTS:

    strategy = experiment["strategy"]
    include_context = experiment["include_context"]

    experiment_name = (
        f"{strategy}_{'with_context' if include_context else 'without_context'}"
    )

    request_file = BATCH_REQUEST_DIR / f"{experiment_name}.jsonl"

    print(f"Uploading {request_file.name}...")

    uploaded_file = client.files.upload(
        file=request_file,
        config={
            "mime_type": "application/jsonl"
        }
    )

    uploaded_files[experiment_name] = uploaded_file

    print(f"✓ Uploaded ({uploaded_file.name})")

Uploading zero_shot_without_context.jsonl...
✓ Uploaded (files/xdmgggiyc3q3)
Uploading zero_shot_with_context.jsonl...
✓ Uploaded (files/wcnc27m051tj)
Uploading zero_shot_cot_without_context.jsonl...
✓ Uploaded (files/03ry14djre01)
Uploading zero_shot_cot_with_context.jsonl...
✓ Uploaded (files/sgm4sst86ady)
Uploading few_shot_cot_without_context.jsonl...
✓ Uploaded (files/0zl0vvv883dh)
Uploading few_shot_cot_with_context.jsonl...
✓ Uploaded (files/g8xtia43zrh2)


In [70]:
uploaded_files.keys()

dict_keys(['zero_shot_without_context', 'zero_shot_with_context', 'zero_shot_cot_without_context', 'zero_shot_cot_with_context', 'few_shot_cot_without_context', 'few_shot_cot_with_context'])

## Create Batch Jobs

Create a Gemini Batch API job for each uploaded request file.

Each experiment is submitted as an independent batch job, allowing all experimental configurations to be processed asynchronously.

The created batch jobs are stored in a dictionary keyed by the experiment name. This enables the remaining stages of the pipeline (monitoring, downloading results, and evaluation) to reference each job efficiently.

In [72]:
batch_jobs = {}

for experiment_name, uploaded_file in uploaded_files.items():

    print(f"Creating batch job for {experiment_name}...")

    batch_job = client.batches.create(
        model=MODEL_NAME,
        src=uploaded_file.name
    )

    batch_jobs[experiment_name] = batch_job

    print(f"✓ Batch Job Created")
    print(f"  Job Name : {batch_job.name}")
    print(f"  State    : {batch_job.state.name}")
    print()

Creating batch job for zero_shot_without_context...
✓ Batch Job Created
  Job Name : batches/d8zoo5t08gygj06xuxhtrfxyfy3qtgehg9xv
  State    : JOB_STATE_PENDING

Creating batch job for zero_shot_with_context...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}